In [ ]:
# Upload this notebook into Colab directly. Keep REPO_DIR pointed at the folder in Google Drive.
from google.colab import drive
drive.mount('/content/drive')

REPO_DIR = "/content/drive/MyDrive/colab_hc3_bundle"


In [ ]:
%cd "$REPO_DIR"
!python -m pip install -q --upgrade pip
!python -m pip install -q --upgrade -r requirements.txt


In [ ]:
import sys
from pathlib import Path

sys.path.append(REPO_DIR)

import torch
from colab_pipeline import (
    ColabExperimentConfig,
    run_baseline_reference,
    run_gptzero_experiment,
)

print("cuda:", torch.cuda.is_available())
print("gpu:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)


Baseline models. This section scores the existing SVM and XGBoost models on HC3. Use the next cell as-is to use the saved baseline models, or enable the optional retraining cell after it if you want fresh baseline models.

In [ ]:
baseline_config = ColabExperimentConfig(
    data_dir=Path(REPO_DIR) / "artifacts" / "data" / "hc3",
    baseline_model_dir=Path(REPO_DIR) / "artifacts" / "models" / "baselines",
    baseline_run_dir=Path(REPO_DIR) / "artifacts" / "runs" / "hc3_baselines_recheck",
    row_batch_size=1024,
    target_fpr=0.01,
)

baseline_result = run_baseline_reference(
    config=baseline_config,
    retrain=False,
    score_splits=("test",),
)
baseline_result


In [ ]:
retrain_baselines = False

if retrain_baselines:
    baseline_retrain_config = ColabExperimentConfig(
        data_dir=Path(REPO_DIR) / "artifacts" / "data" / "hc3",
        baseline_model_dir=Path(REPO_DIR) / "artifacts" / "models" / "baselines_retrained",
        baseline_run_dir=Path(REPO_DIR) / "artifacts" / "runs" / "hc3_baselines_retrained",
        row_batch_size=1024,
        xgb_batch_size=1024,
        xgb_device="cuda",
        target_fpr=0.01,
    )
    baseline_retrain_result = run_baseline_reference(
        config=baseline_retrain_config,
        retrain=True,
        score_splits=("test",),
    )
    baseline_retrain_result
else:
    print("Skipping baseline retraining")


Optional: download a local GPT-2 scorer copy. Run this once with `REPO_DIR` pointing at this local folder(CSCI544-FinalProject\ZeroGPT) if you want `hf_models/gpt2/` populated before uploading the folder to Google Drive.

In [ ]:
download_lm_dir = Path(REPO_DIR) / "hf_models" / "gpt2"
if download_lm_dir.exists():
    print(f"Local GPTZero scorer model already present at {download_lm_dir}")
else:
    from transformers import AutoModelForCausalLM, AutoTokenizer

    download_lm_dir.mkdir(parents=True, exist_ok=True)
    tokenizer = AutoTokenizer.from_pretrained("gpt2")
    tokenizer.save_pretrained(download_lm_dir)
    scorer_model = AutoModelForCausalLM.from_pretrained("gpt2", use_safetensors=False)
    scorer_model.save_pretrained(download_lm_dir, safe_serialization=False)
    print(f"Saved local GPTZero scorer model to {download_lm_dir}")


GPTZero-like model. This section trains the GPTZero-like detector on HC3, then scores the test split. It prefers the local `hf_models/gpt2/` copy when present, and falls back to downloading `gpt2` only if that folder is missing.

In [ ]:
local_lm_dir = Path(REPO_DIR) / "hf_models" / "gpt2"
if local_lm_dir.exists():
    lm_model = str(local_lm_dir)
    local_files_only = True
    print(f"Using local GPTZero scorer model: {lm_model}")
else:
    lm_model = "gpt2"
    local_files_only = False
    print("Local hf_models/gpt2 not found. Falling back to Hugging Face download for gpt2.")

gptzero_config = ColabExperimentConfig(
    data_dir=Path(REPO_DIR) / "artifacts" / "data" / "hc3",
    gptzero_model_dir=Path(REPO_DIR) / "artifacts" / "models" / "gptzero_like",
    gptzero_run_dir=Path(REPO_DIR) / "artifacts" / "runs" / "hc3_gptzero_run",
    lm_model=lm_model,
    device="cuda",
    local_files_only=local_files_only,
    row_batch_size=64,
    perplexity_batch_size=16,
    max_sentences_per_text=None,
    target_fpr=0.01,
)

gptzero_result = run_gptzero_experiment(
    config=gptzero_config,
    train_model=True,
    score_splits=("test",),
)
gptzero_result


Plotting. These cells load the metrics from the baseline and GPTZero-like runs and generate comparison figures for ROC behavior and summary metrics.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

baseline_metrics_dir = baseline_config.baseline_run_dir / "metrics"
gptzero_metrics_dir = gptzero_config.gptzero_run_dir / "metrics"

baseline_summary = pd.read_csv(baseline_metrics_dir / "metrics_summary.csv")
baseline_roc = pd.read_csv(baseline_metrics_dir / "roc_points.csv")
gptzero_summary = pd.read_csv(gptzero_metrics_dir / "metrics_summary.csv")
gptzero_roc = pd.read_csv(gptzero_metrics_dir / "roc_points.csv")

summary = pd.concat([baseline_summary, gptzero_summary], ignore_index=True)
roc = pd.concat([baseline_roc, gptzero_roc], ignore_index=True)
summary_test = summary[summary["split"] == "test"].copy()
roc_test = roc[roc["split"] == "test"].copy()


In [ ]:
plt.figure(figsize=(7, 6))

for detector_name, group in roc_test.groupby("detector_name"):
    group = group.sort_values("fpr")
    auc_value = summary_test.loc[summary_test["detector_name"] == detector_name, "roc_auc"].iloc[0]
    plt.semilogx(group["fpr"].clip(lower=1e-5), group["tpr"], linewidth=2, label=f"{detector_name} (AUC={auc_value:.4f})")

plt.xlim(1e-4, 1.0)
plt.ylim(0.0, 1.001)
plt.xlabel("False Positive Rate (log scale)")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve on HC3 Test Split")
plt.legend()
plt.grid(alpha=0.3, which="both")
plt.tight_layout()
plt.show()


In [ ]:
metrics_to_plot = ["accuracy", "f1", "roc_auc", "tpr_at_1pct_fpr"]
labels = ["Accuracy", "F1", "ROC-AUC", "TPR@1%FPR"]
detectors = summary_test["detector_name"].tolist()
x = np.arange(len(detectors))
width = 0.18

fig, ax = plt.subplots(figsize=(10, 6))
for i, (metric, label) in enumerate(zip(metrics_to_plot, labels)):
    ax.bar(x + (i - 1.5) * width, summary_test[metric], width, label=label)

ax.set_xticks(x)
ax.set_xticklabels(detectors, rotation=15)
ax.set_ylim(0, 1.05)
ax.set_ylabel("Score")
ax.set_title("Test Metrics by Detector")
ax.legend()
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()
